# L6: Multi-agent Collaboration for Financial Analysis

In this lesson, you will learn ways for making agents collaborate with each other.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai crewai[tools] langchain_community langchain_openai
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [2]:
from crewai import Agent, Task, Crew

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [3]:
import os
from utils import get_openai_api_key, get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## crewAI Tools

In [4]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

## Creating Agents

In [5]:
data_analyst_agent = Agent(
    role="Data Analyst",
    goal="Monitor and analyze market data in real-time "
         "to identify trends and predict market movements.",
    backstory="Specializing in financial markets, this agent "
              "uses statistical modeling and machine learning "
              "to provide crucial insights. With a knack for data, "
              "the Data Analyst Agent is the cornerstone for "
              "informing trading decisions.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [6]:
trading_strategy_agent = Agent(
    role="Trading Strategy Developer",
    goal="Develop and test various trading strategies based "
         "on insights from the Data Analyst Agent.",
    backstory="Equipped with a deep understanding of financial "
              "markets and quantitative analysis, this agent "
              "devises and refines trading strategies. It evaluates "
              "the performance of different approaches to determine "
              "the most profitable and risk-averse options.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [7]:
execution_agent = Agent(
    role="Trade Advisor",
    goal="Suggest optimal trade execution strategies "
         "based on approved trading strategies.",
    backstory="This agent specializes in analyzing the timing, price, "
              "and logistical details of potential trades. By evaluating "
              "these factors, it provides well-founded suggestions for "
              "when and how trades should be executed to maximize "
              "efficiency and adherence to strategy.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [8]:
risk_management_agent = Agent(
    role="Risk Advisor",
    goal="Evaluate and provide insights on the risks "
         "associated with potential trading activities.",
    backstory="Armed with a deep understanding of risk assessment models "
              "and market dynamics, this agent scrutinizes the potential "
              "risks of proposed trades. It offers a detailed analysis of "
              "risk exposure and suggests safeguards to ensure that "
              "trading activities align with the firm’s risk tolerance.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

## Creating Tasks

In [9]:
# Task for Data Analyst Agent: Analyze Market Data
data_analysis_task = Task(
    description=(
        "Continuously monitor and analyze market data for "
        "the selected stock ({stock_selection}). "
        "Use statistical modeling and machine learning to "
        "identify trends and predict market movements."
    ),
    expected_output=(
        "Insights and alerts about significant market "
        "opportunities or threats for {stock_selection}."
    ),
    agent=data_analyst_agent,
)

In [10]:
# Task for Trading Strategy Agent: Develop Trading Strategies
strategy_development_task = Task(
    description=(
        "Develop and refine trading strategies based on "
        "the insights from the Data Analyst and "
        "user-defined risk tolerance ({risk_tolerance}). "
        "Consider trading preferences ({trading_strategy_preference})."
    ),
    expected_output=(
        "A set of potential trading strategies for {stock_selection} "
        "that align with the user's risk tolerance."
    ),
    agent=trading_strategy_agent,
)


In [11]:
# Task for Trade Advisor Agent: Plan Trade Execution
execution_planning_task = Task(
    description=(
        "Analyze approved trading strategies to determine the "
        "best execution methods for {stock_selection}, "
        "considering current market conditions and optimal pricing."
    ),
    expected_output=(
        "Detailed execution plans suggesting how and when to "
        "execute trades for {stock_selection}."
    ),
    agent=execution_agent,
)


In [12]:
# Task for Risk Advisor Agent: Assess Trading Risks
risk_assessment_task = Task(
    description=(
        "Evaluate the risks associated with the proposed trading "
        "strategies and execution plans for {stock_selection}. "
        "Provide a detailed analysis of potential risks "
        "and suggest mitigation strategies."
    ),
    expected_output=(
        "A comprehensive risk analysis report detailing potential "
        "risks and mitigation recommendations for {stock_selection}."
    ),
    agent=risk_management_agent,
)

## Creating the Crew
- The `Process` class helps to delegate the workflow to the Agents (kind of like a Manager at work)
- In the example below, it will run this hierarchically.
- `manager_llm` lets you choose the "manager" LLM you want to use.

In [13]:
from crewai import Crew, Process
from langchain_openai import ChatOpenAI

# Define the crew with agents and tasks
financial_trading_crew = Crew(
    agents=[data_analyst_agent, 
            trading_strategy_agent, 
            execution_agent, 
            risk_management_agent],
    
    tasks=[data_analysis_task, 
           strategy_development_task, 
           execution_planning_task, 
           risk_assessment_task],
    
    manager_llm=ChatOpenAI(model="gpt-4-turbo", 
                           temperature=0.7),
    process=Process.hierarchical,
    verbose=True
)

## Running the Crew

- Set the inputs for the execution of the crew.

In [14]:
# Example data for kicking off the process
financial_trading_inputs = {
    'stock_selection': 'TESLA',
    'initial_capital': '100000',
    'risk_tolerance': 'Medium',
    'trading_strategy_preference': 'Long-Term Investing',
    'news_impact_consideration': True
}

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [15]:
### this execution will take some time to run
result = financial_trading_crew.kickoff(inputs=financial_trading_inputs)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 034cfab2-583d-4bd8-a951-76f16a580871                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Continuously monitor and analyze market data for the selected stock (TESLA). Use statistical modeling    │
│  and machine learning to identify trends and predict market movements.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Analyst                                                                                            │
│                                                                                                                 │
│  Task: Continuously monitor and analyze market data for Tesla using statistical modeling and machine learning   │
│  to identify trends and predict market movements.                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Analyst                                                                                            │
│                                                                                                                 │
│  Thought: Thought: To begin analyzing Tesla's stock performance, I need to gather the most up-to-date market    │
│  data available online, including price movements and trading volumes.                                          │
│                                                                                                                 │
│  Using Tool: Search the internet with Serper                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "search_query": "Tesla stock price today"                                                                    │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Comprehensive Risk Analysis Report for Tesla Inc. Trading Strategies and Execution Plans:                      │
│                                                                                                                 │
│  **1. Financial Risks:**                                                                                        │
│     - **Compensation Packages:**                                                                                │
│       - Elon Musk's compensation packages pose a significant financial risk. The Delaware Supreme Court's       │
│  pending decision on Musk's previous compensation package could lead to a substantial $26 billion charge to     │
│  Tesla's profits over two years if the appeal fails. This scenario would drastically impact Tesla's net         │
│  income, considering that $26 billion is more than half of Tesla's total net income since becoming profitable   │
│  in 2019.                                                                                                       │
│       - Even if the appeal is successful, the potential for large payouts if Musk meets certain performance     │
│  thresholds could strain Tesla's profits for years to come.                                                     │
│                                                                                                                 │
│  **2. Legal Risks:**                                                                                            │
│     - **Ongoing Litigation:**                                                                                   │
│       - The outcome of the litigation surrounding Musk's compensation package is uncertain and presents a       │
│  legal risk that could have immediate financial implications for Tesla. A negative outcome could also affect    │
│  investor confidence and Tesla's stock price.                                                                   │
│                                                                                                                 │
│  **3. Stock Volatility:**                                                                                       │
│     - **High Beta Value:**                                                                                      │
│       - Tesla's beta of 1.90 indicates a volatility level nearly twice that of the market. This high            │
│  volatility increases the investment risk, particularly in a dynamic market environment.                        │
│                                                                                                                 │
│  **4. Operational Risks:**                                                                                      │
│     - **Technological and Market Expansion:**                                                                   │
│       - While Tesla's expansion into new markets and technological innovations (e.g., autonomous vehicles,      │
│  ride-hail service in Arizona) present growth opportunities, they also introduce risks related to execution     │
│  and market acceptance.                                                                                         │
│                                                                                                                 │
│  **5. Mitigation Strategies:**                         

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 66fa1c8a-4b97-4f46-9187-d3a7de2e4fe2                                                                     │
│  Agent: Crew Manager                                                                                            │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 034cfab2-583d-4bd8-a951-76f16a580871                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: Comprehensive Risk Analysis Report for Tesla Inc. Trading Strategies and Execution Plans:        │
│                                                                                                                 │
│  **1. Financial Risks:**                                                                                        │
│     - **Compensation Packages:**                                                                                │
│       - Elon Musk's compensation packages pose a significant financial risk. The Delaware Supreme Court's       │
│  pending decision on Musk's previous compensation package could lead to a substantial $26 billion charge to     │
│  Tesla's profits over two years if the appeal fails. This scenario would drastically impact Tesla's net         │
│  income, considering that $26 billion is more than half of Tesla's total net income since becoming profitable   │
│  in 2019.                                                                                                       │
│       - Even if the appeal is successful, the potential for large payouts if Musk meets certain performance     │
│  thresholds could strain Tesla's profits for years to come.                                                     │
│                                                                                                                 │
│  **2. Legal Risks:**                                                                                            │
│     - **Ongoing Litigation:**                                                                                   │
│       - The outcome of the litigation surrounding Musk's compensation package is uncertain and presents a       │
│  legal risk that could have immediate financial implications for Tesla. A negative outcome could also affect    │
│  investor confidence and Tesla's stock price.                                                                   │
│                                                                                                                 │
│  **3. Stock Volatility:**                                                                                       │
│     - **High Beta Value:**                                                                                      │
│       - Tesla's beta of 1.90 indicates a volatility level nearly twice that of the market. This high            │
│  volatility increases the investment risk, particularly in a dynamic market environment.                        │
│                                                                                                                 │
│  **4. Operational Risks:**                                                                                      │
│     - **Technological and Market Expansion:**                                                                   │
│       - While Tesla's expansion into new markets and technological innovations (e.g., autonomous vehicles,      │
│  ride-hail service in Arizona) present growth opportunities, they also introduce risks related to execution     │
│  and market acceptance.                                                                                         │
│                                                       

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



┌───────────────────────────── Execution Traces ──────────────────────────────┐
│                                                                             │
│  🔍 Detailed execution traces are available!                                │
│                                                                             │
│  View insights including:                                                   │
│    • Agent decision-making process                                          │
│    • Task execution flow and timing                                         │
│    • Tool usage details                                                     │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
Would you like to view your execution traces? [y/N] (20s timeout): 

┌───────────────────────── Tracing Preference Saved ──────────────────────────┐
│                                                 

- Display the final result as Markdown.

In [19]:
from IPython.display import Markdown, display
display(Markdown(result.raw))

Comprehensive Risk Analysis Report for Tesla Inc. Trading Strategies and Execution Plans:

**1. Financial Risks:**
   - **Compensation Packages:**
     - Elon Musk's compensation packages pose a significant financial risk. The Delaware Supreme Court's pending decision on Musk's previous compensation package could lead to a substantial $26 billion charge to Tesla's profits over two years if the appeal fails. This scenario would drastically impact Tesla's net income, considering that $26 billion is more than half of Tesla's total net income since becoming profitable in 2019.
     - Even if the appeal is successful, the potential for large payouts if Musk meets certain performance thresholds could strain Tesla's profits for years to come.

**2. Legal Risks:**
   - **Ongoing Litigation:**
     - The outcome of the litigation surrounding Musk's compensation package is uncertain and presents a legal risk that could have immediate financial implications for Tesla. A negative outcome could also affect investor confidence and Tesla's stock price.

**3. Stock Volatility:**
   - **High Beta Value:**
     - Tesla's beta of 1.90 indicates a volatility level nearly twice that of the market. This high volatility increases the investment risk, particularly in a dynamic market environment.

**4. Operational Risks:**
   - **Technological and Market Expansion:**
     - While Tesla's expansion into new markets and technological innovations (e.g., autonomous vehicles, ride-hail service in Arizona) present growth opportunities, they also introduce risks related to execution and market acceptance.

**5. Mitigation Strategies:**
   - **Strategic Financial Planning:**
     - Implement diversified investment strategies to mitigate the impacts of stock volatility and potential financial setbacks from Musk's compensation packages.
   - **Legal and Compliance Vigilance:**
     - Closely monitor ongoing legal proceedings and prepare for various outcomes to manage potential impacts swiftly.
   - **Risk Management in Trading:**
     - Use strategic stop-loss orders and periodic reassessment of the stock based on latest financial and market data.
   - **Leverage Market Research:**
     - Continuously monitor market trends and analyst forecasts to adjust trading strategies accordingly.

**Conclusion:**
This report outlines key risks associated with trading Tesla's stock, focusing on financial, legal, and operational challenges. By implementing the suggested mitigation strategies, Tesla can manage these risks effectively, aligning with its medium risk tolerance and long-term investment objectives.